# Advanced RAG Patterns: a mini pipeline from scratch

**Goal:** build, step by step, the main *advanced* patterns from the Day 10 lecture on a small knowledge base, and **measure** what each one gives you.

```
docs -> chunk + context -> dense + BM25 indexes -> filter -> hybrid (RRF) -> re-rank -> small-to-big -> grounded prompt
        (Steps 4-5)        (Step 6)                (7)       (8-10)         (11)       (12)           (14-15)

query -> [multi-query / HyDE] (Step 13)                    evaluation: Recall@k, MRR (Steps 16-17), latency (Step 18)
```

| Limitation from the lecture | Symptom | Pattern used in this notebook |
|---|---|---|
| Retrieval irrelevance | X100 text returned for an X200 question | context-enriched chunks, **hybrid search (BM25 + dense, RRF)**, **re-ranking** |
| Residual hallucinations | outdated 2022 refund policy mixed with the 2024 one | **metadata filter** (date), **grounded prompt** with citations |
| Security | a confidential finance note shown to everybody | **access filter in the retrieval layer** |
| Chunking | answer cut in the middle of an idea | fixed-length vs sentence chunks, **small-to-big** retrieval |
| Short or vague questions | paraphrase does not match the document wording | **multi-query**, **HyDE** |
| Performance lag | slow, expensive prompts | re-rank then cut top-k, latency per stage |

**How to work with this notebook**
* Cells marked **TODO** are yours: replace `# TODO: place your code here`.
* Cells marked **Check** contain assertions - do not edit them.
* Cells without a marker are provided.
* The notebook needs **no API key**: the LLM calls of the lecture (query rewriting, HyDE, answer generation) are replaced by fixed texts, so everything runs on a free Colab CPU.

## Step 0. Install dependencies (provided)

In [1]:
!pip install -q sentence-transformers rank-bm25

## Step 1. Imports - TODO
Import:
* `re` and `time`
* `Dict`, `List`, `Optional`, `Sequence` and `Tuple` from `typing`
* `numpy` as `np`
* `BM25Okapi` from `rank_bm25`
* `CrossEncoder` and `SentenceTransformer` from `sentence_transformers`

In [2]:
import os
import re
import time
import warnings
from typing import Dict, List, Optional, Sequence, Tuple

# hide download warnings and progress bars so the outputs stay readable
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore")

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer

## Step 2. Load the models - TODO
Two different model types are used in a two-stage retrieval (slides 31-32):

* `embedder` - the **bi-encoder** `sentence-transformers/all-MiniLM-L6-v2` (first stage, fast, document vectors are pre-computed),
* `reranker` - the **cross-encoder** `cross-encoder/ms-marco-MiniLM-L-6-v2` with `max_length=512` (second stage, slower, more precise).

In [3]:
# fast model for the first search, slower and more exact one for the re-ranking
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)

In [4]:
# Check
assert isinstance(embedder, SentenceTransformer)
assert isinstance(reranker, CrossEncoder)
print("✅ both models loaded")

✅ both models loaded


## Step 3. The knowledge base (provided)
Eight small documents, each with **metadata** (slide 27): `doc_type`, `valid_from` (year) and `access` (who may read it).
Notice the traps built into the data:
* the **2022 and 2024 refund policies** are almost identical but contradict each other (30 vs 14 days),
* **X100 and X200** warranty texts look alike and differ only in one digit,
* **E-4711** is an exact identifier - embeddings handle such tokens poorly,
* one document is **confidential** (`access = "finance"`) and must never reach an ordinary user.

The first line of every text is its title, paragraphs are separated by a blank line.

In [5]:
docs = [
    {"id": "refund_policy_2022",
     "metadata": {"doc_type": "policy", "valid_from": 2022, "access": "all"},
     "text": "Refund policy (2022 edition)\n\n"
             "Annual plans can be returned within 30 days after purchase. The refund is paid to the original payment method.\n\n"
             "Monthly plans cannot be refunded after the billing date. Customers may cancel at any time to stop future charges."},
    {"id": "refund_policy_2024",
     "metadata": {"doc_type": "policy", "valid_from": 2024, "access": "all"},
     "text": "Refund policy (2024 edition)\n\n"
             "Annual plans can be returned within 14 days after purchase. The refund is paid to the original payment method within 5 business days.\n\n"
             "Monthly plans cannot be refunded after the billing date. Customers may cancel at any time to stop future charges.\n\n"
             "Enterprise contracts follow the terms of the individual agreement and are not covered by this policy."},
    {"id": "warranty_x100",
     "metadata": {"doc_type": "manual", "valid_from": 2024, "access": "all"},
     "text": "X100 laptop warranty\n\n"
             "The X100 laptop has a warranty of 12 months from the delivery date. The warranty covers hardware defects but not accidental damage.\n\n"
             "To claim the warranty, contact support and provide the serial number printed on the bottom of the device."},
    {"id": "warranty_x200",
     "metadata": {"doc_type": "manual", "valid_from": 2024, "access": "all"},
     "text": "X200 laptop warranty\n\n"
             "The X200 laptop has a warranty of 24 months from the delivery date. The warranty covers hardware defects but not accidental damage.\n\n"
             "The X200 battery is covered for 6 months only. To claim the warranty, contact support and provide the serial number printed on the bottom of the device."},
    {"id": "billing_troubleshooting",
     "metadata": {"doc_type": "kb", "valid_from": 2024, "access": "all"},
     "text": "Billing troubleshooting guide\n\n"
             "Error E-4711 means that the invoice generation failed because the billing address is missing. Add a complete billing address in the account settings and retry.\n\n"
             "Error E-5020 means that the payment card was declined by the bank. Ask the customer to use another card or contact the bank."},
    {"id": "shipping",
     "metadata": {"doc_type": "policy", "valid_from": 2024, "access": "all"},
     "text": "Shipping and delivery\n\n"
             "Standard delivery takes 3 to 5 business days within the European Union. Express delivery takes 1 to 2 business days and costs extra.\n\n"
             "Orders outside the European Union can take up to 14 days because of customs checks."},
    {"id": "finance_refund_note",
     "metadata": {"doc_type": "internal", "valid_from": 2024, "access": "finance"},
     "text": "Internal finance note on refunds\n\n"
             "Refunds above 500 EUR must be approved by the finance director. Key accounts received an exception: their annual plans can be returned within 60 days.\n\n"
             "This note is confidential and must not be shared with customers."},
    {"id": "password_help",
     "metadata": {"doc_type": "guide", "valid_from": 2024, "access": "all"},
     "text": "Account and password help\n\n"
             "To reset your password, open the login page and choose the option for a forgotten password. A reset link is sent to your registered email address.\n\n"
             "For security reasons the link expires after 30 minutes."},
]
print(len(docs), "documents")

8 documents


## Step 4. Fixed-length chunking - TODO (slides 20-21, 25)
Implement `fixed_length_chunks(text, chunk_size, overlap)`. It must:
1. split the text into words (`text.split()`),
2. cut chunks of `chunk_size` words, where each next chunk starts `chunk_size - overlap` words after the previous one (so neighbouring chunks **share `overlap` words**),
3. stop as soon as a chunk reaches the end of the text (no tiny tail chunk consisting only of overlap),
4. raise `ValueError` if `overlap >= chunk_size`, and return `[]` for an empty text.

In [6]:
def fixed_length_chunks(text: str, chunk_size: int = 20, overlap: int = 5) -> List[str]:
    """Split `text` into chunks of `chunk_size` words; neighbouring chunks share `overlap` words."""
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    words = text.split()
    step = chunk_size - overlap
    pieces = []
    start = 0
    while start < len(words):
        pieces.append(" ".join(words[start:start + chunk_size]))
        # this chunk already reaches the end, so no tail made only of overlap
        if start + chunk_size >= len(words):
            break
        start += step
    return pieces

In [7]:
# Check
_text = " ".join(f"w{i}" for i in range(10))
assert fixed_length_chunks(_text, chunk_size=4, overlap=1) == ["w0 w1 w2 w3", "w3 w4 w5 w6", "w6 w7 w8 w9"]
assert fixed_length_chunks("", 4, 1) == []
assert fixed_length_chunks("a b c", 10, 2) == ["a b c"]
try:
    fixed_length_chunks(_text, 4, 4)
    raise AssertionError("ValueError expected when overlap >= chunk_size")
except ValueError:
    pass
print("✅ fixed_length_chunks works")

✅ fixed_length_chunks works


Look at what fixed-length chunking does to a real document (provided). Where are the borders? Are ideas cut in the middle?

In [8]:
for i, chunk in enumerate(fixed_length_chunks(docs[1]["text"].replace("\n\n", " "), chunk_size=14, overlap=3)):
    print(f"[{i}] {chunk}")

[0] Refund policy (2024 edition) Annual plans can be returned within 14 days after purchase.
[1] days after purchase. The refund is paid to the original payment method within 5
[2] method within 5 business days. Monthly plans cannot be refunded after the billing date.
[3] the billing date. Customers may cancel at any time to stop future charges. Enterprise
[4] future charges. Enterprise contracts follow the terms of the individual agreement and are not
[5] and are not covered by this policy.


## Step 5. Sentence chunks with context + small-to-big parents (provided)
Fixed-length chunks cut through ideas, so the pipeline uses **structure-aware** chunking (recursive idea: *paragraph -> sentence*, slides 23 and 26):

* **child chunk** = one sentence - small, so the match is precise; it is the unit that is embedded and searched,
* **parent** = the whole paragraph it came from - returned to the LLM later (**small-to-big**, slide 26),
* **context-enriched**: the document title is prepended to every child (slide 24), so the sentence *"Annual plans can be returned within 14 days"* knows that it belongs to the *2024 refund policy*.

In [9]:
def split_sentences(text: str) -> List[str]:
    return [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s]


def tokenize(text: str) -> List[str]:
    """Lower-case word tokens; identifiers such as 'e-4711' stay in one piece."""
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())


def build_chunks(docs: List[dict]) -> List[dict]:
    chunks = []
    for doc in docs:
        title, *paragraphs = [p.strip() for p in doc["text"].split("\n\n") if p.strip()]
        for p_idx, paragraph in enumerate(paragraphs):
            for sentence in split_sentences(paragraph):
                chunks.append({
                    "chunk_id": len(chunks),
                    "doc_id": doc["id"],
                    "title": title,
                    "text": f"{title}. {sentence}",          # child: what we embed / search
                    "parent_id": f"{doc['id']}#{p_idx}",
                    "parent": paragraph,                       # parent: what the LLM will read
                    **doc["metadata"],
                })
    return chunks


chunks = build_chunks(docs)
print(len(chunks), "child chunks")
for c in chunks[:3]:
    print(c["chunk_id"], c["doc_id"], "|", c["text"])

29 child chunks
0 refund_policy_2022 | Refund policy (2022 edition). Annual plans can be returned within 30 days after purchase.
1 refund_policy_2022 | Refund policy (2022 edition). The refund is paid to the original payment method.
2 refund_policy_2022 | Refund policy (2022 edition). Monthly plans cannot be refunded after the billing date.


## Step 6. Build the two indexes - TODO
Create

1. `chunk_embeddings` - the matrix of **normalised** embeddings of all `chunk["text"]` values: `embedder.encode(texts, normalize_embeddings=True)`. With normalised vectors the dot product equals the cosine similarity.
2. `bm25` - a `BM25Okapi` index built from `tokenize(chunk["text"])` of every chunk.

In [10]:
chunk_texts = [c["text"] for c in chunks]

chunk_embeddings = embedder.encode(chunk_texts, normalize_embeddings=True)
bm25 = BM25Okapi([tokenize(t) for t in chunk_texts])

In [11]:
# Check
assert chunk_embeddings.shape[0] == len(chunks)
assert np.allclose(np.linalg.norm(chunk_embeddings, axis=1), 1.0, atol=1e-3), "embeddings must be normalised"
assert bm25.corpus_size == len(chunks)
print("✅ indexes built:", chunk_embeddings.shape, "dense matrix +", bm25.corpus_size, "BM25 documents")

✅ indexes built: (29, 384) dense matrix + 29 BM25 documents


## Step 7. Metadata and access filter - TODO (slide 27)
Implement `filter_chunks(chunks, user_groups, min_year, doc_type)`. It returns the **list of `chunk_id`s** that a user may see and that are fresh enough. A chunk is allowed when

* its `access` value is in `user_groups`,
* its `valid_from` is `>= min_year`,
* and, only if `doc_type` is not `None`, its `doc_type` equals `doc_type`.

The filter is applied **before** the search, in the retrieval layer - not by asking the LLM to behave.

In [12]:
def filter_chunks(chunks: List[dict], user_groups: Sequence[str], min_year: int = 0,
                  doc_type: Optional[str] = None) -> List[int]:
    """Return the chunk_ids that the user may see (see the rules above)."""
    visible = []
    for c in chunks:
        if c["access"] not in user_groups:
            continue
        if c["valid_from"] < min_year:
            continue
        if doc_type is not None and c["doc_type"] != doc_type:
            continue
        visible.append(c["chunk_id"])
    return visible

In [13]:
# Check
_fake = [
    {"chunk_id": 0, "access": "all",     "valid_from": 2022, "doc_type": "policy"},
    {"chunk_id": 1, "access": "all",     "valid_from": 2024, "doc_type": "policy"},
    {"chunk_id": 2, "access": "finance", "valid_from": 2024, "doc_type": "internal"},
    {"chunk_id": 3, "access": "all",     "valid_from": 2024, "doc_type": "guide"},
]
assert filter_chunks(_fake, ["all"]) == [0, 1, 3]
assert filter_chunks(_fake, ["all"], min_year=2024) == [1, 3]
assert filter_chunks(_fake, ["all", "finance"], min_year=2024) == [1, 2, 3]
assert filter_chunks(_fake, ["all"], min_year=2024, doc_type="guide") == [3]
print("✅ filter_chunks works")

# the real data: nothing confidential or outdated may remain for an ordinary user
_ids = filter_chunks(chunks, ["all"], min_year=2024)
assert all(chunks[i]["access"] == "all" and chunks[i]["valid_from"] >= 2024 for i in _ids)
print("   ", len(_ids), "of", len(chunks), "chunks are visible to an ordinary user")

✅ filter_chunks works
    22 of 29 chunks are visible to an ordinary user


## Step 8. Dense and BM25 search - TODO (slide 28)
Two retrievers, both returning a **ranked list of `chunk_id`s, best first, at most `k` items**, and both restricted to `allowed` (a list of chunk ids; `None` = no restriction).

* `dense_search` - embed the query with `normalize_embeddings=True`, compute the scores `chunk_embeddings @ query_vector`.
* `bm25_search` - `bm25.get_scores(tokenize(query))`; **drop chunks with score <= 0** (no keyword in common - they carry no information).

The helper `top_k_ids` (provided) sorts the scores and applies the filter.

In [14]:
def top_k_ids(scores, k: int, allowed: Optional[Sequence[int]] = None, min_score: Optional[float] = None) -> List[int]:
    ids = range(len(scores)) if allowed is None else allowed
    if min_score is not None:
        ids = [i for i in ids if scores[i] > min_score]
    return sorted(ids, key=lambda i: scores[i], reverse=True)[:k]

In [15]:
def dense_search(query: str, k: int = 10, allowed: Optional[Sequence[int]] = None) -> List[int]:
    """Cosine-similarity search. Returns chunk_ids, best first."""
    query_vec = embedder.encode(query, normalize_embeddings=True)
    scores = chunk_embeddings @ query_vec
    return top_k_ids(scores, k, allowed)


def bm25_search(query: str, k: int = 10, allowed: Optional[Sequence[int]] = None) -> List[int]:
    """Keyword search. Returns chunk_ids with a positive BM25 score, best first."""
    scores = bm25.get_scores(tokenize(query))
    # score 0 or less means no shared keyword, so skip those chunks
    return top_k_ids(scores, k, allowed, min_score=0)

In [16]:
# Check
_q = "What does error E-4711 mean?"
_d, _b = dense_search(_q, k=5), bm25_search(_q, k=5)
assert len(_d) == 5 and len(set(_d)) == 5 and all(isinstance(i, (int, np.integer)) for i in _d)
assert 1 <= len(_b) <= 5
assert "E-4711" in chunks[_b[0]]["text"], "BM25 must find the exact identifier"
_allowed = [0, 1, 2, 3]
assert set(dense_search(_q, k=10, allowed=_allowed)) <= set(_allowed)
assert set(bm25_search("annual plans refund", k=10, allowed=_allowed)) <= set(_allowed)
print("✅ dense_search and bm25_search work")

✅ dense_search and bm25_search work


Compare the two retrievers on an exact identifier and on a paraphrase (provided). Which one is better for which query?

In [17]:
def show(ids: Sequence[int], n: int = 3) -> None:
    for i in list(ids)[:n]:
        print(f"   #{i:<3} {chunks[i]['text'][:95]}")


for q in ["What does error E-4711 mean?", "How can I get my money back for a yearly plan?"]:
    print("QUERY:", q)
    print(" dense:")
    show(dense_search(q, 3))
    print(" BM25:")
    show(bm25_search(q, 3))
    print()

QUERY: What does error E-4711 mean?
 dense:
   #16  Billing troubleshooting guide. Error E-4711 means that the invoice generation failed because th
   #18  Billing troubleshooting guide. Error E-5020 means that the payment card was declined by the ban
   #19  Billing troubleshooting guide. Ask the customer to use another card or contact the bank.
 BM25:
   #16  Billing troubleshooting guide. Error E-4711 means that the invoice generation failed because th
   #18  Billing troubleshooting guide. Error E-5020 means that the payment card was declined by the ban

QUERY: How can I get my money back for a yearly plan?
 dense:
   #4   Refund policy (2024 edition). Annual plans can be returned within 14 days after purchase.
   #0   Refund policy (2022 edition). Annual plans can be returned within 30 days after purchase.
   #6   Refund policy (2024 edition). Monthly plans cannot be refunded after the billing date.
 BM25:
   #26  Account and password help. To reset your password, open the login p

## Step 9. Reciprocal Rank Fusion - TODO (slides 28-29)
BM25 scores and cosine similarities live on different scales, so RRF merges **ranks**, not scores:

`RRF(doc) = sum over all ranked lists of 1 / (k + rank)`, with `rank` starting at **1** and `k = 60`. A document missing from a list gets nothing from that list.

Implement `reciprocal_rank_fusion(rankings, k=60)`. `rankings` is a list of ranked lists of ids; return a list of `(id, score)` tuples sorted by score, best first.

In [18]:
def reciprocal_rank_fusion(rankings: List[List], k: int = 60) -> List[Tuple]:
    """Fuse several ranked lists. Returns [(id, rrf_score), ...], best first."""
    totals = {}
    for ranking in rankings:
        for rank, item in enumerate(ranking, start=1):
            totals[item] = totals.get(item, 0) + 1 / (k + rank)
    return sorted(totals.items(), key=lambda pair: pair[1], reverse=True)

In [19]:
# Check: the worked example from slide 29
_bm25_list  = ["A", "B", "D", "C"]      # A=1, B=2, D=3, C=4
_dense_list = ["B", "C", "A"]           # B=1, C=2, A=3, D not found
_fused = reciprocal_rank_fusion([_bm25_list, _dense_list])
assert [d for d, _ in _fused] == ["B", "A", "C", "D"], _fused
assert [round(s, 4) for _, s in _fused] == [0.0325, 0.0323, 0.0318, 0.0159], _fused
assert [d for d, _ in reciprocal_rank_fusion([["x", "y", "z"]])] == ["x", "y", "z"]
print("✅ reciprocal_rank_fusion reproduces the lecture example")

✅ reciprocal_rank_fusion reproduces the lecture example


## Step 10. Hybrid search - TODO
Implement `hybrid_search(query, k, allowed, n_candidates)`:
1. take `n_candidates` results from `bm25_search` and from `dense_search` (both with the same `allowed`),
2. fuse the two lists with `reciprocal_rank_fusion`,
3. return the first `k` **chunk ids** (without the scores).

In [20]:
def hybrid_search(query: str, k: int = 10, allowed: Optional[Sequence[int]] = None,
                  n_candidates: int = 30) -> List[int]:
    """BM25 + dense search merged with RRF. Returns the best k chunk_ids."""
    keyword_hits = bm25_search(query, n_candidates, allowed)
    meaning_hits = dense_search(query, n_candidates, allowed)
    fused = reciprocal_rank_fusion([keyword_hits, meaning_hits])
    return [cid for cid, _ in fused[:k]]

In [21]:
# Check
_res = hybrid_search("What does error E-4711 mean?", k=5)
assert 1 <= len(_res) <= 5 and len(set(_res)) == len(_res)
assert "E-4711" in chunks[_res[0]]["text"] or any("E-4711" in chunks[i]["text"] for i in _res)
_allowed = filter_chunks(chunks, ["all"], min_year=2024)
assert set(hybrid_search("refund for annual plans", k=10, allowed=_allowed)) <= set(_allowed)
print("✅ hybrid_search works")

✅ hybrid_search works


### Why the filter matters (provided)
The same refund question, once without and once with the metadata filter for an ordinary user (`access = "all"`, documents from 2024 on). Compare the documents behind the top results.

In [22]:
q = "How many days do I have to return an annual subscription?"
allowed = filter_chunks(chunks, ["all"], min_year=2024)

print("WITHOUT filter:")
show(hybrid_search(q, k=4), n=4)
print("\nWITH filter (access=all, valid_from >= 2024):")
show(hybrid_search(q, k=4, allowed=allowed), n=4)

WITHOUT filter:
   #24  Internal finance note on refunds. Key accounts received an exception: their annual plans can be
   #4   Refund policy (2024 edition). Annual plans can be returned within 14 days after purchase.
   #0   Refund policy (2022 edition). Annual plans can be returned within 30 days after purchase.
   #5   Refund policy (2024 edition). The refund is paid to the original payment method within 5 busine

WITH filter (access=all, valid_from >= 2024):
   #4   Refund policy (2024 edition). Annual plans can be returned within 14 days after purchase.
   #5   Refund policy (2024 edition). The refund is paid to the original payment method within 5 busine
   #7   Refund policy (2024 edition). Customers may cancel at any time to stop future charges.
   #21  Shipping and delivery. Express delivery takes 1 to 2 business days and costs extra.


## Step 11. Cross-encoder re-ranking - TODO (slides 30-32)
First stage = **recall** (fast), second stage = **precision** (slow, one model call per candidate).

Implement `rerank(model, query, chunk_ids, top_k)`. It must
1. build one `[query, chunk text]` pair per candidate,
2. score the pairs with `model.predict(pairs, show_progress_bar=False)`,
3. return the best `top_k` as a list of `(chunk_id, float(score))`, best first.

In [23]:
def rerank(model, query: str, chunk_ids: Sequence[int], top_k: int = 5) -> List[Tuple[int, float]]:
    """Re-order `chunk_ids` by cross-encoder score. Returns [(chunk_id, score), ...]."""
    if len(chunk_ids) == 0:
        return []

    pairs = [[query, chunks[cid]["text"]] for cid in chunk_ids]
    scores = model.predict(pairs, show_progress_bar=False)

    scored = [(cid, float(s)) for cid, s in zip(chunk_ids, scores)]
    scored.sort(key=lambda pair: pair[1], reverse=True)
    return scored[:top_k]

In [24]:
# Check: a fake model makes the test independent of the real one ("longer text = more relevant")
class _FakeCrossEncoder:
    def predict(self, pairs, **kwargs):
        return [len(text) for _, text in pairs]

_ids = list(range(8))
_res = rerank(_FakeCrossEncoder(), "q", _ids, top_k=3)
_expected = sorted(_ids, key=lambda i: -len(chunks[i]["text"]))[:3]
assert [c for c, _ in _res] == _expected, _res
assert isinstance(_res[0][1], float)
assert rerank(_FakeCrossEncoder(), "q", [], top_k=3) == []
print("✅ rerank works")

✅ rerank works


How does re-ranking change the order of the hybrid candidates? (provided)

In [25]:
q = "How long is the warranty for the X200?"
candidates = hybrid_search(q, k=8, allowed=allowed)
print("hybrid order (RRF):")
show(candidates, n=4)
print("\nafter cross-encoder re-ranking:")
for cid, score in rerank(reranker, q, candidates, top_k=4):
    print(f"   #{cid:<3} {score:7.3f}  {chunks[cid]['text'][:85]}")

hybrid order (RRF):
   #14  X200 laptop warranty. The X200 battery is covered for 6 months only.
   #12  X200 laptop warranty. The X200 laptop has a warranty of 24 months from the delivery date.
   #13  X200 laptop warranty. The warranty covers hardware defects but not accidental damage.
   #15  X200 laptop warranty. To claim the warranty, contact support and provide the serial number prin

after cross-encoder re-ranking:


   #12    9.388  X200 laptop warranty. The X200 laptop has a warranty of 24 months from the delivery d
   #14    8.586  X200 laptop warranty. The X200 battery is covered for 6 months only.
   #9     5.913  X100 laptop warranty. The X100 laptop has a warranty of 12 months from the delivery d
   #13    1.974  X200 laptop warranty. The warranty covers hardware defects but not accidental damage.


## Step 12. Small-to-big: return the parent - TODO (slide 26)
We searched with small, precise **children**; the LLM should read the **parents** (full paragraphs).

Implement `to_parents(chunk_ids)`. For every chunk id, take its parent; return a list of dicts `{"parent_id", "doc_id", "title", "text"}` where `text` is the chunk's `"parent"` value. **Deduplicate** by `parent_id`, keeping the order of the first appearance (the best-ranked child decides the parent's position).

In [26]:
def to_parents(chunk_ids: Sequence[int]) -> List[dict]:
    """Replace children by their (deduplicated) parents."""
    parents = []
    seen = set()
    for cid in chunk_ids:
        c = chunks[cid]
        if c["parent_id"] in seen:
            continue
        seen.add(c["parent_id"])
        parents.append({
            "parent_id": c["parent_id"],
            "doc_id": c["doc_id"],
            "title": c["title"],
            "text": c["parent"],
        })
    return parents

In [27]:
# Check
_same_parent = [c["chunk_id"] for c in chunks if c["parent_id"] == chunks[0]["parent_id"]]
assert len(_same_parent) >= 2, "the data should contain a paragraph with several sentences"
_res = to_parents(_same_parent)
assert len(_res) == 1 and _res[0]["text"] == chunks[0]["parent"]
assert set(_res[0]) == {"parent_id", "doc_id", "title", "text"}
_mixed = [chunks[0]["chunk_id"], chunks[-1]["chunk_id"], chunks[0]["chunk_id"]]
assert [p["parent_id"] for p in to_parents(_mixed)] == [chunks[0]["parent_id"], chunks[-1]["parent_id"]]
assert to_parents([]) == []
print("✅ to_parents works")

✅ to_parents works


## Step 13. Query transformations: multi-query and HyDE - TODO (slides 33-34)
In production an **LLM** writes the paraphrases and the hypothetical answer. To keep this notebook free of API keys they are given as fixed texts (provided below). The retrieval part is yours.

In [28]:
original_query = "How can I get my money back for a yearly plan?"

# what an LLM would return for "write 3 paraphrases of the question"
paraphrases = [
    original_query,
    "refund period for an annual subscription",
    "money-back guarantee for a yearly plan",
    "cancel annual plan and get a refund",
]

# what an LLM would return for "write a short passage that answers the question" (HyDE); it may contain wrong facts
hypothetical_answer = (
    "Annual subscriptions can be refunded within a set number of days after purchase. "
    "The refund is paid back to the original payment method."
)

Implement

* `multi_query_search(queries, k, allowed)` - run `hybrid_search` for **every** query, fuse all result lists with `reciprocal_rank_fusion`, return the best `k` chunk ids,
* `hyde_search(hypothetical_text, k, allowed)` - embed the **hypothetical answer instead of the question** and run the dense search with it. The hypothetical text is only a search aid and is never shown to the user.

In [29]:
def multi_query_search(queries: List[str], k: int = 10, allowed: Optional[Sequence[int]] = None) -> List[int]:
    """Hybrid search for every query, fused with RRF. Returns the best k chunk_ids."""
    all_lists = [hybrid_search(q, k, allowed) for q in queries]
    fused = reciprocal_rank_fusion(all_lists)
    return [cid for cid, _ in fused[:k]]


def hyde_search(hypothetical_text: str, k: int = 10, allowed: Optional[Sequence[int]] = None) -> List[int]:
    """Dense search with a hypothetical answer instead of the question."""
    # the fake answer looks more like the real document than the question does
    return dense_search(hypothetical_text, k, allowed)

In [30]:
# Check
assert multi_query_search([original_query], k=5, allowed=allowed) == hybrid_search(original_query, k=5, allowed=allowed)
_mq = multi_query_search(paraphrases, k=6, allowed=allowed)
assert 1 <= len(_mq) <= 6 and len(set(_mq)) == len(_mq) and set(_mq) <= set(allowed)
_hy = hyde_search(hypothetical_answer, k=6, allowed=allowed)
assert len(_hy) == 6 and set(_hy) <= set(allowed)
assert _hy == dense_search(hypothetical_answer, k=6, allowed=allowed)
print("✅ multi_query_search and hyde_search work")

✅ multi_query_search and hyde_search work


Compare the three ways of searching for the same vague question (provided). Do the top documents differ?

In [31]:
print("QUERY:", original_query)
print("\n plain hybrid search:")
show(hybrid_search(original_query, 3, allowed))
print("\n multi-query + RRF:")
show(multi_query_search(paraphrases, 3, allowed))
print("\n HyDE (dense search with the hypothetical answer):")
show(hyde_search(hypothetical_answer, 3, allowed))

QUERY: How can I get my money back for a yearly plan?

 plain hybrid search:
   #4   Refund policy (2024 edition). Annual plans can be returned within 14 days after purchase.
   #26  Account and password help. To reset your password, open the login page and choose the option fo
   #17  Billing troubleshooting guide. Add a complete billing address in the account settings and retry

 multi-query + RRF:


   #4   Refund policy (2024 edition). Annual plans can be returned within 14 days after purchase.
   #7   Refund policy (2024 edition). Customers may cancel at any time to stop future charges.
   #26  Account and password help. To reset your password, open the login page and choose the option fo

 HyDE (dense search with the hypothetical answer):
   #5   Refund policy (2024 edition). The refund is paid to the original payment method within 5 busine
   #4   Refund policy (2024 edition). Annual plans can be returned within 14 days after purchase.
   #7   Refund policy (2024 edition). Customers may cancel at any time to stop future charges.


## Step 14. Grounded prompt - TODO (slide 35)
Implement `build_prompt(question, parents)`. The prompt must contain, in this order:

1. the instruction `Answer ONLY from the context below.`
2. the permission to refuse: `If the answer is not in the context, say "I don't know".`
3. the citation rule `Cite sources as [1], [2].`
4. a blank line, the word `Context:` and **numbered** parents, one per line, in the form `[1] <title>: <text>` (numbering starts at 1),
5. a blank line and `Question: <question>`.

In [32]:
def build_prompt(question: str, parents: List[dict]) -> str:
    """Instruction + numbered context + question."""
    lines = [
        "Answer ONLY from the context below.",
        'If the answer is not in the context, say "I don\'t know".',
        "Cite sources as [1], [2].",
        "",
        "Context:",
    ]
    for num, p in enumerate(parents, start=1):
        lines.append(f"[{num}] {p['title']}: {p['text']}")
    lines += ["", f"Question: {question}"]
    return "\n".join(lines)

In [33]:
# Check
_p = [{"parent_id": "a#0", "doc_id": "a", "title": "Title A", "text": "Text A."},
      {"parent_id": "b#0", "doc_id": "b", "title": "Title B", "text": "Text B."}]
_prompt = build_prompt("Why?", _p)
assert _prompt.startswith("Answer ONLY from the context below.")
assert "I don't know" in _prompt and "[1], [2]" in _prompt
assert "[1] Title A: Text A." in _prompt and "[2] Title B: Text B." in _prompt
assert _prompt.index("[1] Title A") < _prompt.index("[2] Title B") < _prompt.index("Question: Why?")
print("✅ build_prompt works")

✅ build_prompt works


## Step 15. Put the pipeline together - TODO
Implement `advanced_rag(question, user_groups, min_year, n_candidates, top_k)`:

1. `allowed = filter_chunks(...)`
2. `candidates = hybrid_search(question, k=n_candidates, allowed=allowed)`
3. `reranked = rerank(reranker, question, candidates, top_k=top_k)`
4. `parents = to_parents(...)` of the re-ranked chunk ids
5. return `(build_prompt(question, parents), parents)`

In [34]:
def advanced_rag(question: str, user_groups: Sequence[str], min_year: int = 2024,
                 n_candidates: int = 30, top_k: int = 3) -> Tuple[str, List[dict]]:
    """filter -> hybrid search -> re-rank -> small-to-big -> grounded prompt."""
    allowed = filter_chunks(chunks, user_groups, min_year)
    candidates = hybrid_search(question, k=n_candidates, allowed=allowed)
    reranked = rerank(reranker, question, candidates, top_k=top_k)
    parents = to_parents([cid for cid, _ in reranked])
    return build_prompt(question, parents), parents

In [35]:
# Check
_prompt, _parents = advanced_rag("How many days do I have to return an annual subscription?", ["all"])
assert 1 <= len(_parents) <= 3
assert "Question: How many days" in _prompt and _prompt.count("[1]") >= 1
assert all(p["doc_id"] != "finance_refund_note" for p in _parents), "confidential document leaked!"
assert all(p["doc_id"] != "refund_policy_2022" for p in _parents), "outdated document used!"
assert "finance director" not in _prompt
print("✅ advanced_rag works - no confidential or outdated text in the prompt")

✅ advanced_rag works - no confidential or outdated text in the prompt


Run the pipeline for an ordinary user and for a finance employee (provided). The prompt is what you would now send to an LLM.

In [36]:
question = "How many days do I have to return an annual subscription?"

prompt, parents = advanced_rag(question, user_groups=["all"])
print(prompt)

print("\n" + "=" * 70)
print("Same question, finance employee (groups: all + finance):\n")
prompt_fin, _ = advanced_rag(question, user_groups=["all", "finance"])
print(prompt_fin)

# Optional (needs an API key): send `prompt` to an LLM of your choice and check that the answer cites [1].

Answer ONLY from the context below.
If the answer is not in the context, say "I don't know".
Cite sources as [1], [2].

Context:
[1] Refund policy (2024 edition): Annual plans can be returned within 14 days after purchase. The refund is paid to the original payment method within 5 business days.
[2] Refund policy (2024 edition): Monthly plans cannot be refunded after the billing date. Customers may cancel at any time to stop future charges.

Question: How many days do I have to return an annual subscription?

Same question, finance employee (groups: all + finance):



Answer ONLY from the context below.
If the answer is not in the context, say "I don't know".
Cite sources as [1], [2].

Context:
[1] Refund policy (2024 edition): Annual plans can be returned within 14 days after purchase. The refund is paid to the original payment method within 5 business days.
[2] Internal finance note on refunds: Refunds above 500 EUR must be approved by the finance director. Key accounts received an exception: their annual plans can be returned within 60 days.

Question: How many days do I have to return an annual subscription?

## Step 16. Evaluation: Recall@k and MRR - TODO (slide 36)
"Without evaluation every change is a guess." A tiny **golden set** (provided) maps each question to the document that contains its answer. We evaluate on the level of **documents**: the ranked chunk ids are converted to ranked doc ids (duplicates removed, order kept) by `to_doc_ids` (provided).

Implement

* `recall_at_k(ranked_docs, relevant, k)` - the share of the `relevant` set (a `set` of doc ids) that appears in the first `k` ranked docs,
* `reciprocal_rank(ranked_docs, relevant)` - `1 / position` (starting at 1) of the first relevant document, `0.0` if there is none,
* `evaluate(search_fn, golden, k)` - for every `(query, relevant)` pair call `search_fn(query)` (it returns chunk ids), convert them with `to_doc_ids`, and return the **mean Recall@k** and the **mean MRR** as a tuple.

In [37]:
golden = [
    ("How many days do I have to return an annual subscription?", {"refund_policy_2024"}),
    ("How can I get my money back for a yearly plan?",           {"refund_policy_2024"}),
    ("How long is the warranty for the X200?",                   {"warranty_x200"}),
    ("What does error E-4711 mean?",                             {"billing_troubleshooting"}),
    ("How long does delivery take?",                             {"shipping"}),
    ("I forgot my password, how do I get back into my account?", {"password_help"}),
]


def to_doc_ids(chunk_ids: Sequence[int]) -> List[str]:
    ranked = []
    for cid in chunk_ids:
        doc_id = chunks[cid]["doc_id"]
        if doc_id not in ranked:
            ranked.append(doc_id)
    return ranked

In [38]:
def recall_at_k(ranked_docs: List[str], relevant: set, k: int) -> float:
    """Share of the relevant documents found in the first k."""
    if not relevant:
        return 0.0
    found = relevant & set(ranked_docs[:k])
    return len(found) / len(relevant)


def reciprocal_rank(ranked_docs: List[str], relevant: set) -> float:
    """1 / position of the first relevant document (0.0 if none)."""
    for pos, doc_id in enumerate(ranked_docs, start=1):
        if doc_id in relevant:
            return 1 / pos
    return 0.0


def evaluate(search_fn, golden: List[Tuple[str, set]], k: int = 3) -> Tuple[float, float]:
    """Return (mean Recall@k, mean MRR) of `search_fn` over the golden set."""
    recalls, rr_values = [], []
    for query, relevant in golden:
        ranked_docs = to_doc_ids(search_fn(query))
        recalls.append(recall_at_k(ranked_docs, relevant, k))
        rr_values.append(reciprocal_rank(ranked_docs, relevant))
    return float(np.mean(recalls)), float(np.mean(rr_values))

In [39]:
# Check
assert recall_at_k(["a", "b", "c"], {"b"}, k=1) == 0.0
assert recall_at_k(["a", "b", "c"], {"b"}, k=2) == 1.0
assert recall_at_k(["a", "b", "c"], {"b", "z"}, k=3) == 0.5
assert reciprocal_rank(["a", "b", "c"], {"b"}) == 0.5
assert reciprocal_rank(["a", "b", "c"], {"x"}) == 0.0

_fake_search = {"q1": [0], "q2": [len(chunks) - 1]}                # chunk 0 -> doc 0, last chunk -> last doc
_gold = [("q1", {chunks[0]["doc_id"]}), ("q2", {"does-not-exist"})]
_r, _m = evaluate(lambda q: _fake_search[q], _gold, k=3)
assert (_r, _m) == (0.5, 0.5), (_r, _m)
print("✅ metrics work")

✅ metrics work


## Step 17. Compare the configurations - TODO
The dictionary `configs` maps a name to a function `query -> ranked chunk ids` (up to 10). Four configurations are provided. **Add a fifth one**, `"hybrid + filter + re-rank"`: hybrid search with `k=30` and the filter, then `rerank(reranker, query, ..., top_k=10)`, returning only the chunk ids.

In [40]:
ALLOWED = filter_chunks(chunks, ["all"], min_year=2024)       # an ordinary user, current documents only

configs = {
    "dense only":              lambda q: dense_search(q, 10),
    "BM25 only":               lambda q: bm25_search(q, 10),
    "hybrid (RRF)":            lambda q: hybrid_search(q, 10),
    "hybrid + filter":         lambda q: hybrid_search(q, 10, ALLOWED),
}

In [41]:
def filtered_reranked(query):
    candidates = hybrid_search(query, 30, ALLOWED)
    return [cid for cid, _ in rerank(reranker, query, candidates, top_k=10)]


configs["hybrid + filter + re-rank"] = filtered_reranked

In [42]:
# Check
assert "hybrid + filter + re-rank" in configs
assert len(configs["hybrid + filter + re-rank"](golden[0][0])) <= 10
print("✅ five configurations ready")

✅ five configurations ready


Run the comparison (provided). Read the table, then answer the questions at the end.

In [43]:
print(f"{'configuration':<28}{'Recall@3':>10}{'MRR':>8}")
print("-" * 46)
for name, search_fn in configs.items():
    recall, mrr = evaluate(search_fn, golden, k=3)
    print(f"{name:<28}{recall:>10.2f}{mrr:>8.2f}")

configuration                 Recall@3     MRR
----------------------------------------------


dense only                        1.00    1.00
BM25 only                         0.83    0.76
hybrid (RRF)                      1.00    0.92
hybrid + filter                   1.00    1.00


hybrid + filter + re-rank         1.00    1.00


## Step 18. Bonus: latency per stage - TODO (slides 13 and 43)
"Measure first, then optimise the slowest stage." Implement `time_stage(fn, repeats)` - call `fn()` `repeats` times, measure with `time.perf_counter()` and return the **average time in milliseconds**. The loop below (provided) uses it for every stage.

In [44]:
def time_stage(fn, repeats: int = 5) -> float:
    """Average run time of fn() in milliseconds."""
    start = time.perf_counter()
    for _ in range(repeats):
        fn()
    return (time.perf_counter() - start) * 1000 / repeats

In [45]:
# Check
assert 0 <= time_stage(lambda: sum(range(1000)), repeats=3) < 1000
print("✅ time_stage works")

✅ time_stage works


In [46]:
q = "How long is the warranty for the X200?"
cands = hybrid_search(q, 30, ALLOWED)

stages = {
    "metadata filter":           lambda: filter_chunks(chunks, ["all"], 2024),
    "embed query + dense":       lambda: dense_search(q, 30, ALLOWED),
    "BM25":                      lambda: bm25_search(q, 30, ALLOWED),
    "hybrid (BM25+dense+RRF)":   lambda: hybrid_search(q, 30, ALLOWED),
    "cross-encoder re-rank":     lambda: rerank(reranker, q, cands, top_k=3),
}
for name, fn in stages.items():
    print(f"{name:<28}{time_stage(fn):8.1f} ms")

metadata filter                  0.0 ms
embed query + dense             11.5 ms
BM25                             0.2 ms
hybrid (BM25+dense+RRF)         14.0 ms


cross-encoder re-rank          153.1 ms


### Questions (answer in a text cell below)
1. **Chunking.** What is wrong with the fixed-length chunks printed in Step 4? Why does the sentence chunk with the **title prepended** help the query *"How long is the warranty for the X200?"*? Try removing the title from `"text"` in `build_chunks` - what changes?
2. **Dense vs BM25.** Which queries did each retriever handle better in Step 8 and why? Why is *E-4711* hard for embeddings?
3. **RRF.** Why does RRF use ranks instead of scores? What would go wrong if you simply added a BM25 score and a cosine similarity?
4. **Filters.** Look at the two outputs in Step 10. Which document won without the filter and which with it? Why should the access filter live in the retrieval layer and not in the prompt?
5. **Re-ranking.** Did the cross-encoder change the order of the hybrid candidates in Step 11? Which stage is the slowest in Step 18, and what happens to latency if `n_candidates` grows from 30 to 300?
6. **Query transformations.** Compare plain, multi-query and HyDE in Step 13. When would you pay the extra LLM call?
7. **Evaluation.** Read the table of Step 17: which technique gave the biggest gain on this golden set? Why can a 6-question set not prove that a technique is better, and how big should a real golden set be?
8. **Extension.** Ask `advanced_rag` a question that the documents cannot answer (for example *"What is the CEO's salary?"*). Read the prompt it builds: what protects the system from a hallucinated answer, and what would you add (slide 12)?

In [47]:
# question 1: search the X200 question with and without the title in the chunk text
q = "How long is the warranty for the X200?"
plain_texts = [c["text"].split(". ", 1)[1] for c in chunks]   # drop "<title>. " from the front
plain_emb = embedder.encode(plain_texts, normalize_embeddings=True)
q_vec = embedder.encode(q, normalize_embeddings=True)

print("with title:")
show(top_k_ids(chunk_embeddings @ q_vec, 3))
print("\nwithout title:")
for i in top_k_ids(plain_emb @ q_vec, 3):
    print(f"   #{i:<3} {plain_texts[i][:95]}")

with title:
   #12  X200 laptop warranty. The X200 laptop has a warranty of 24 months from the delivery date.
   #14  X200 laptop warranty. The X200 battery is covered for 6 months only.
   #15  X200 laptop warranty. To claim the warranty, contact support and provide the serial number prin

without title:
   #12  The X200 laptop has a warranty of 24 months from the delivery date.
   #14  The X200 battery is covered for 6 months only.
   #9   The X100 laptop has a warranty of 12 months from the delivery date.


In [48]:
# question 3: raw scores of both retrievers for the same query
q = "What does error E-4711 mean?"
bm25_raw = bm25.get_scores(tokenize(q))
cos_raw = chunk_embeddings @ embedder.encode(q, normalize_embeddings=True)

for i in top_k_ids(bm25_raw, 3):
    print(f"#{i:<3} BM25 {bm25_raw[i]:6.2f}   cosine {cos_raw[i]:5.2f}")

#16  BM25   5.11   cosine  0.66
#18  BM25   2.43   cosine  0.56
#0   BM25   0.00   cosine  0.06


In [49]:
# question 8: a question the documents can't answer
prompt_ceo, parents_ceo = advanced_rag("What is the CEO's salary?", user_groups=["all"])
print(prompt_ceo)

Answer ONLY from the context below.
If the answer is not in the context, say "I don't know".
Cite sources as [1], [2].

Context:
[1] Shipping and delivery: Standard delivery takes 3 to 5 business days within the European Union. Express delivery takes 1 to 2 business days and costs extra.
[2] Refund policy (2024 edition): Annual plans can be returned within 14 days after purchase. The refund is paid to the original payment method within 5 business days.

Question: What is the CEO's salary?


### My answers

1. Chunking. The fixed-length chunks cut sentences in the middle. "within 14" ends one chunk and "days after purchase" starts the next, and the same happens with "within 5 / business days". A chunk like "and are not covered by this policy." means nothing alone. With the title in front, every sentence says which product it is about. "The warranty covers hardware defects but not accidental damage" is in both the X100 and X200 texts, and only the title tells them apart. When I removed the title, the top 2 results stayed the same, but the third one became the X100 sentence (#9). With the title, all top 3 were from the X200 document.

2. Dense vs BM25. For E-4711 both found the right chunk first, but BM25 only returned the two error chunks, while dense also put the E-5020 chunk second because it looks almost the same as a vector. For "money back for a yearly plan" dense was much better: it found the annual plan refund chunks even though the words are different. BM25 returned password and battery chunks because there is no "refund" or "annual" in the question. Embeddings are bad with E-4711 because the model splits it into small pieces and the number has no meaning for it, so E-4711 and E-5020 end up close.

3. RRF. The two scores live on different scales. In my check for the E-4711 query, BM25 gave 5.11 and 2.43 to the top two chunks, while cosine gave 0.66 and 0.56. If I just add them, the BM25 gap of 2.68 decides everything and the cosine gap of 0.10 hardly counts. Ranks are on the same scale for both lists, so RRF can mix them fairly without any tuning.

4. Filters. Without the filter, the confidential finance note (60 days) was first and the old 2022 policy (30 days) was third. With it, the 2024 policy (14 days) was first. If the filter is only in the prompt, the secret text still reaches the LLM and a tricky question can make it leak. If retrieval removes it, the model never sees it.

5. Re-ranking. Yes. RRF had the battery sentence first, and the cross-encoder moved the "24 months" sentence to the top, which is the real answer, although it also pulled the X100 sentence up to third. It is the slowest stage by far, about 150 ms against about 14 ms for the whole hybrid search. It runs once per candidate, so 300 candidates instead of 30 would be about 10 times slower (around 1.5 s).

6. Query transformations. Plain hybrid search had the right chunk first but then password and billing chunks. Multi-query added one more refund chunk to the top 3. HyDE gave three chunks from the 2024 refund policy, so it was the best here. I would pay for the extra LLM call when questions are short or vague, or when users use other words than the documents. For exact things like error codes it is not needed, and BM25 already does it.

7. Evaluation. The filter gave the biggest gain: hybrid MRR went from 0.92 to 1.00, because without it the finance note came first for the refund question. Dense alone also got 1.00, and BM25 was the weakest (0.83 and 0.76). Re-ranking did not change the numbers because they were already at the maximum. With 6 questions, one question changes recall by 0.17, so a difference can be just luck. A real golden set should have at least a few hundred questions, with different types (exact codes, paraphrases, questions with no answer).

8. Extension. For "What is the CEO's salary?" the pipeline still gives the LLM the shipping and refund paragraphs, because retrieval always returns something. The protection is only in the prompt: answer only from the context, say "I don't know", and cite sources. I would also add a minimum re-rank score, so if nothing relevant is found the system says "I don't know" without calling the LLM, and a check after generation that every sentence of the answer has a citation that really supports it.